# Setup

In [1]:
#load basic libraries
import polars as pl  # It is advised to use polars, as the detasets can be quite memory-intensive
import numpy as np
import torch.nn as nn
from torch import optim
import torch
import re
from tqdm import trange
import random
from utils import *
from data.simulate_walk_the_book import *
import warnings



%load_ext autoreload
%autoreload 2
%matplotlib inline

In [18]:
dataset = "BTCUSDT"
data_root = "data"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

data_direct = f"{data_root}/{dataset}"
X_train = pl.read_parquet(f"{data_direct}/X_train.parquet")
y_train = pl.read_parquet(f"{data_direct}/y_train.parquet")

y_train_raw = y_train.__copy__()

# load the volume to fill info from the text file
with open(f"{data_direct}/vol_to_fill.txt", "r") as file:
    content = file.read().strip()

match = re.search(r"The volume to fill is: ([\d.]+)", content)
if match:
    volume_to_fill  = float(match.group(1))
    print(f"Extracted number: {volume_to_fill}")
else:
    print("No number found in the file.")


Extracted number: 4.0


In [19]:
X_train, y_train, means, stds, X_id_map, y_id_map = preprocess(X_train, y_train, device)

# id_map contains the mapping from index along the second dimension to anonymized_id used in the tensor
# X_train, y_train have shape (seq_len, num_ids, num_features)

# TWAP

In [ ]:
# Get ID values
id_values = X_id_map[:, 1].cpu().numpy()

# id column
ids_repeated = np.repeat(id_values, 60)

# time_in_hour column
times_seconds = np.arange(59*60, 59*60 + 60) * 1000
unique_ids = X_train.shape[1]
times_repeated = np.tile(times_seconds, unique_ids)

twap_preds = pl.DataFrame({
    'anonymized_id': ids_repeated,
    'time_in_hour': times_repeated,
    'position': volume_to_fill / 60 # Ensure volume is flat
}).with_columns(
    # This cast turns raw integers into the Duration type that prints as '59m 1s'
    pl.col('time_in_hour').cast(pl.Duration(time_unit='ms'))
)

twap_preds

# TWAP + last k seconds

In [11]:
ks = [60, 30, 15, 5]
id_values = X_id_map[:, 1].cpu().numpy()
unique_ids = len(id_values)

# Setup the base structure (IDs and Times are constant)
ids_repeated = np.repeat(id_values, 60)
times_seconds = np.arange(59*60, 59*60 + 60) * 1000
times_repeated = np.tile(times_seconds, unique_ids)

# Dictionary to store the different TWAP variants
twap_variants = {}

for k in ks:
    # 1. Create the position pattern for one ID
    # Zeros for the first (60-k) seconds, then volume/k for the last k seconds
    single_id_position = np.concatenate([
        np.zeros(60 - k),
        np.full(k, volume_to_fill / k)
    ])
    
    # 2. Tile this pattern for all unique IDs
    positions_repeated = np.tile(single_id_position, unique_ids)
    
    # 3. Build the DataFrame
    df_name = f"twap_{k}"
    twap_variants[df_name] = pl.DataFrame({
        'anonymized_id': ids_repeated,
        'time_in_hour': times_repeated,
        'position': positions_repeated
    }).with_columns(
        pl.col('time_in_hour').cast(pl.Duration(time_unit='ms'))
    )


# Computing Basis Points

When walking the book, we use unmodified y_train.

> bps scores are different than in the example as we're dropping some anonymized ids during data cleaning.

In [ ]:
compute_bps(twap_preds, y_train_raw, volume_to_fill)

In [12]:
warnings.filterwarnings("ignore")

# Define the variants
model_types = ["twap_60", "twap_30", "twap_15", "twap_5"]
data_root = "data"

results_accumulator = []

for dataset in DATASETS:
    # 1. Load data ONCE per dataset to save time
    data_direct = f"{data_root}/{dataset}"
    X_train_raw = pl.read_parquet(f"{data_direct}/X_train.parquet")
    y_train_orig = pl.read_parquet(f"{data_direct}/y_train.parquet")
    
    # Extract volume to fill
    with open(f"{data_direct}/vol_to_fill.txt", "r") as file:
        content = file.read().strip()
    match = re.search(r"The volume to fill is: ([\d.]+)", content)
    volume_to_fill = float(match.group(1))

    # Preprocess once per dataset
    X_train, y_train, means, stds, X_id_map, y_id_map = preprocess(X_train_raw, y_train_orig, device)
    
    # Get ID values and setup shared time column
    id_values = X_id_map[:, 1].cpu().numpy()
    unique_ids = len(id_values)
    ids_repeated = np.repeat(id_values, 60)
    times_seconds = np.arange(59*60, 59*60 + 60) * 1000
    times_repeated = np.tile(times_seconds, unique_ids)

    for model_type in model_types:
        # 2. Extract k from the model name (e.g., 'twap_30' -> 30)
        k = int(model_type.split('_')[1])
        
        # 3. Create position: 0 for (60-k) rows, then volume/k for the last k rows
        single_id_pos = np.concatenate([
            np.zeros(60 - k),
            np.full(k, volume_to_fill / k)
        ])
        positions_repeated = np.tile(single_id_pos, unique_ids)

        # 4. Create the prediction DataFrame
        twap_preds = pl.DataFrame({
            'anonymized_id': ids_repeated,
            'time_in_hour': times_repeated,
            'position': positions_repeated
        }).with_columns(
            pl.col('time_in_hour').cast(pl.Duration(time_unit='ms'))
        )

        # 5. Compute score (using y_train_orig which is the raw Polars DF)
        bps_score = compute_bps(twap_preds, y_train_orig, volume_to_fill)

        results_accumulator.append({
            "data asset": dataset,
            "model": model_type,
            "basis points": bps_score
        })

# 6. Final Evaluation Table
model_eval_df = pl.DataFrame(results_accumulator)
model_eval_df

data asset,model,basis points
str,str,f64
"""ADAUSDT""","""twap_60""",5.835181
"""ADAUSDT""","""twap_30""",4.543049
"""ADAUSDT""","""twap_15""",3.577797
"""ADAUSDT""","""twap_5""",3.179883
"""BTCUSDT""","""twap_60""",1.496524
…,…,…
"""SOLUSDT""","""twap_5""",5.856552
"""XRPUSDT""","""twap_60""",3.882433
"""XRPUSDT""","""twap_30""",3.33129


In [14]:
model_eval_df.write_csv("twap.csv")

# Ridge

this is done in the command line from Maaz.

In [27]:
from baselines import *

args = {
    "train_fraction": 0.8,
    "alpha": [0.1, 1.0, 10.0]
}

# Pipeline (accumulate this as you go)

Here, we define a pipeline where we create all relevant metrics across all datasets.

we define the following fields
- data asset
- model
- basis points

In [16]:
X_train

ask_price_1,ask_vol_1,bid_price_1,bid_vol_1,ask_price_2,ask_vol_2,bid_price_2,bid_vol_2,ask_price_3,ask_vol_3,bid_price_3,bid_vol_3,ask_price_4,ask_vol_4,bid_price_4,bid_vol_4,ask_price_5,ask_vol_5,bid_price_5,bid_vol_5,open,high,low,close,volume,anonymized_id,time_in_hour
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,u64,duration[ms]
56892.528,0.649013,56892.4741,0.520792,56892.5819,0.00234,56892.0968,0.001528,56894.0911,0.004734,56891.9351,0.000405,56895.1152,0.00374,56891.3961,0.03,56895.2769,0.06988,56889.7252,0.050882,null,null,null,null,null,3442073104987193344,0ms
56892.528,0.7094365,56877.7055,0.350604,56900.1279,0.097365,56875.73815,0.00071,56900.491725,0.080226,56884.6047,0.042489,56904.183875,0.081072,56882.79905,0.032681,56903.6853,0.001578,56879.1069,0.0,56892.528,56892.528,56892.528,56892.528,0.001545,3442073104987193344,1s
56894.28514,0.567232,56892.4741,0.768055,56900.96874,0.020947,56877.579733,0.071667,56904.41295,0.081071,56869.279133,0.245209,56904.5477,0.022358,56871.4531,0.0,56913.4412,0.164116,56858.571,0.020296,null,null,null,null,null,3442073104987193344,2s
56892.528,0.7090038,56891.00802,0.6482008,56898.79118,0.152273,56884.083667,0.388231,56901.86348,0.2499924,56882.089367,0.065192,56905.17294,0.078377,56880.1849,0.08308,56907.4583,0.298564,56875.0105,0.076982,56892.528,56892.528,56892.528,56892.528,0.000047,3442073104987193344,3s
56892.528,0.702552,56887.17034,0.676458,56899.521525,0.1019585,56866.831175,0.000756,56899.99315,0.0194685,56863.354625,0.003211,56899.517033,0.001041,56866.09005,0.0073815,56899.768567,0.000947,56873.2318,0.001209,56892.528,56892.528,56892.528,56892.528,0.00811,3442073104987193344,4s
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
138798.08255,0.633836,138774.00906,0.522962,138818.024875,0.120264,138762.27912,0.0579514,138810.045567,0.001713,138748.7797,0.185772,138814.5644,0.241866,138748.589433,0.054681,138817.91785,0.4645715,138740.5031,0.000267,138794.5864,138794.5864,138794.5864,138794.5864,0.000085,16504566399903900334,58m 56s
138805.51722,0.339833,138778.51838,0.5075852,138806.74444,0.0568954,138766.4745,0.038163,138815.06385,0.116978,138748.46576,0.093335,138815.670325,0.120933,138737.87742,0.099492,138810.4261,0.0,138729.91476,0.1850554,null,null,null,null,null,16504566399903900334,58m 57s
138807.7148,0.3528276,138780.1737,0.5707414,138812.62368,0.1149126,138771.7544,0.1883888,138822.38436,0.3505066,138760.1957,0.121641,138826.35142,0.189758,138752.37574,0.0927846,138840.73558,0.547329,138734.88072,0.1863318,null,null,null,null,null,16504566399903900334,58m 58s
